# Tujuan Modeling dan Aturan Anti-Leakage

Notebook ini HANYA memanggil fungsi dari `src/modeling/train.py`. Tidak ada
definisi fungsi/class di sini. Seluruh model memakai holdout split 80%
development / 20% test dan Stratified 5-Fold CV yang identik, di-freeze ke
`data/processed/holdout_split.json` dan `cv_folds.json`.

In [1]:
import sys
import numpy as np
import pandas as pd
sys.path.append("../src")

from modeling.train import (
    set_random_seed,
    load_featured_dataset,
    load_feature_metadata,
    create_or_load_holdout_split,
    create_or_load_cv_folds,
    RareCategoryGrouper,
    build_linear_mlp_preprocessor,
    build_tree_preprocessor,
    compute_scale_pos_weight,
    evaluate_predictions,
    get_model_dir,
    save_model_pipeline,
    save_model_json,
    update_model_summary,
    train_logistic_regression,
    train_decision_tree,
    train_random_forest,
    train_lightgbm,
    train_xgboost,
    train_catboost,
    train_mlp,
    display_model_comparison,
)

# Load Dataset dan Metadata Fitur

Set random seed global, lalu load dataset featured (307.511 baris, 73 kolom)
dan metadata daftar fitur tree vs linear/MLP.

In [2]:
set_random_seed()
df = load_featured_dataset()
metadata = load_feature_metadata()

print(df.shape)
print(list(metadata.keys()))

(307511, 73)
['tree_features', 'linear_mlp_features', 'n_tree_features', 'n_linear_mlp_features', 'decision_log']


# Buat/Muat Holdout Split dan Stratified CV Fold

Idempotent: kalau `holdout_split.json`/`cv_folds.json` sudah ada, langsung
di-load agar seluruh model memakai sampel identik.

In [3]:
split = create_or_load_holdout_split(df)
folds = create_or_load_cv_folds(df, split)

print(split["development_size"], split["test_size_count"])
print(list(folds.keys()))

246008 61503
['random_seed', 'n_splits', 'development_ids', 'fold_assignment', 'created_at']


# Verifikasi Distribusi Target

Proporsi TARGET=1 harus konsisten (~8,07%) di development set, test set,
dan setiap fold CV — bukti bahwa stratifikasi bekerja.

In [4]:
dev_df = df[df["SK_ID_CURR"].isin(split["development_ids"])]
test_df = df[df["SK_ID_CURR"].isin(split["test_ids"])]

print("Development:", dev_df["TARGET"].value_counts(normalize=True).round(4).to_dict())
print("Test:", test_df["TARGET"].value_counts(normalize=True).round(4).to_dict())

fold_arr = np.array(folds["fold_assignment"])
dev_ids_arr = np.array(folds["development_ids"])

for f in range(folds["n_splits"]):
    fold_ids = dev_ids_arr[fold_arr == f]
    fold_target = df[df["SK_ID_CURR"].isin(fold_ids)]["TARGET"]
    print(f"Fold {f}: n={len(fold_ids)}, proporsi TARGET=1={fold_target.mean():.4f}")

Development: {0: 0.9193, 1: 0.0807}
Test: {0: 0.9193, 1: 0.0807}
Fold 0: n=49202, proporsi TARGET=1=0.0807
Fold 1: n=49202, proporsi TARGET=1=0.0807
Fold 2: n=49202, proporsi TARGET=1=0.0807
Fold 3: n=49201, proporsi TARGET=1=0.0807
Fold 4: n=49201, proporsi TARGET=1=0.0807


# Sanity Check Preprocessing Pipeline

Uji `build_linear_mlp_preprocessor` dan `build_tree_preprocessor` pada
fold-train pertama saja, memastikan jumlah fitur setelah encoding wajar.

In [5]:
fold0_train_ids = dev_ids_arr[fold_arr != 0]

X_linear = df[df["SK_ID_CURR"].isin(fold0_train_ids)][metadata["linear_mlp_features"]]
linear_preprocessor = build_linear_mlp_preprocessor(metadata["linear_mlp_features"])
X_linear_transformed = linear_preprocessor.fit_transform(X_linear)
print("Linear/MLP -> sebelum:", len(metadata["linear_mlp_features"]), "sesudah:", X_linear_transformed.shape[1])

X_tree = df[df["SK_ID_CURR"].isin(fold0_train_ids)][metadata["tree_features"]]
tree_preprocessor = build_tree_preprocessor(metadata["tree_features"])
X_tree_transformed = tree_preprocessor.fit_transform(X_tree)
print("Tree -> sebelum:", len(metadata["tree_features"]), "sesudah:", X_tree_transformed.shape[1])

Linear/MLP -> sebelum: 66 sesudah: 182
Tree -> sebelum: 66 sesudah: 182


# Verifikasi Cepat: compute_scale_pos_weight dan evaluate_predictions

Uji dua fungsi metrik dengan data dummy sederhana sebelum dipakai pada
model sungguhan.

In [6]:
y_dummy_true = [0, 0, 0, 1, 0, 1, 0, 0, 1, 0]
y_dummy_proba = [0.1, 0.2, 0.4, 0.8, 0.3, 0.6, 0.1, 0.05, 0.9, 0.2]

scale_pos_weight_demo = compute_scale_pos_weight(y_dummy_true)
metrics_demo = evaluate_predictions(y_dummy_true, y_dummy_proba)

print("scale_pos_weight:", scale_pos_weight_demo)
print(metrics_demo)

scale_pos_weight: 2.3333333333333335
{'roc_auc': 1.0, 'pr_auc': 1.0, 'log_loss': 0.24151316867743727, 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'confusion_matrix': [[7, 0], [0, 3]], 'threshold': 0.5}


# Verifikasi Cepat: Penyimpanan Artifact dan Model Summary

Uji `save_model_pipeline`, `save_model_json`, dan `update_model_summary`
dengan model dummy sederhana (bukan model sungguhan), lalu bersihkan
artifact test-nya supaya tidak mengotori folder `models/` dan
`reports/model_summary.csv`.

In [7]:
from sklearn.linear_model import LogisticRegression

dummy_model = LogisticRegression()
dummy_model.fit([[0], [1], [0], [1]], [0, 1, 0, 1])

pipeline_path = save_model_pipeline(dummy_model, model_name="test_dummy")
history_path = save_model_json({"note": "dummy history"}, model_name="test_dummy", suffix="history")
metrics_path = save_model_json(metrics_demo, model_name="test_dummy", suffix="metrics")
config_path = save_model_json({"note": "dummy config"}, model_name="test_dummy", suffix="config")

update_model_summary({
    "model_name": "test_dummy",
    "run_id": "test_001",
    "status": "test_only",
    "roc_auc": metrics_demo["roc_auc"],
})

print("Pipeline:", pipeline_path)
print("History:", history_path)
print("Metrics:", metrics_path)
print("Config:", config_path)


print(pd.read_csv("../reports/model_summary.csv"))

Pipeline: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\test_dummy\test_dummy_pipeline.joblib
History: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\test_dummy\test_dummy_history.json
Metrics: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\test_dummy\test_dummy_metrics.json
Config: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\models\test_dummy\test_dummy_config.json
   model_name    run_id     status  feature_path  n_features_before_encoding  \
0  test_dummy  test_001  test_only           NaN                         NaN   

   n_features_after_encoding  cv_roc_auc_mean  cv_roc_auc_std  cv_pr_auc_mean  \
0                        NaN              NaN             NaN             NaN   

   cv_pr_auc_std  best_iteration_mean  training_time_seconds  random_seed  \
0            NaN                  NaN                    NaN          NaN   

   model_path  history_path  timestamp  
0         NaN     

# (Cleanup, jalankan setelah Cell 14 sukses):
Kenapa Ada Cell Cleanup ini
test_dummy bukan model sungguhan — cuma dipakai untuk memverifikasi fungsi save/update bekerja.


In [8]:
import shutil

shutil.rmtree(get_model_dir("test_dummy"))

summary = pd.read_csv("../reports/model_summary.csv")
summary = summary[summary["model_name"] != "test_dummy"]
summary.to_csv("../reports/model_summary.csv", index=False)

print("Artifact test_dummy sudah dibersihkan.")

Artifact test_dummy sudah dibersihkan.


# Trainer Logistic Regression (Baseline Pertama)

Melatih Logistic Regression dengan Stratified 5-Fold CV, retrain pada
seluruh development set, simpan artifact, dan update model summary.

In [9]:
result_lr = train_logistic_regression(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_lr["cv_summary"]["roc_auc_mean"], result_lr["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_lr["cv_summary"]["pr_auc_mean"], result_lr["cv_summary"]["pr_auc_std"]
))

CV ROC-AUC: 0.7475 ± 0.0020
CV PR-AUC: 0.2248 ± 0.0060


# Decision Tree

In [10]:
result_dt = train_decision_tree(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_dt["cv_summary"]["roc_auc_mean"], result_dt["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_dt["cv_summary"]["pr_auc_mean"], result_dt["cv_summary"]["pr_auc_std"]
))

CV ROC-AUC: 0.5388 ± 0.0052
CV PR-AUC: 0.0912 ± 0.0020


# Random Forest

In [11]:
result_rf = train_random_forest(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_rf["cv_summary"]["roc_auc_mean"], result_rf["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_rf["cv_summary"]["pr_auc_mean"], result_rf["cv_summary"]["pr_auc_std"]
))

CV ROC-AUC: 0.7441 ± 0.0021
CV PR-AUC: 0.2187 ± 0.0038


In [12]:
shutil.rmtree(get_model_dir("lightgbm"))
summary = pd.read_csv("../reports/model_summary.csv")
summary = summary[summary["model_name"] != "lightgbm"]
summary.to_csv("../reports/model_summary.csv", index=False)

# Lightgbm

In [13]:
result_lgbm = train_lightgbm(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_lgbm["cv_summary"]["roc_auc_mean"], result_lgbm["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_lgbm["cv_summary"]["pr_auc_mean"], result_lgbm["cv_summary"]["pr_auc_std"]
))
print("Rata-rata best_iteration:", np.mean([fm["best_iteration"] for fm in result_lgbm["cv_fold_metrics"]]))



c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\USER\miniconda3\envs\Homecreadit\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set =

CV ROC-AUC: 0.7571 ± 0.0010
CV PR-AUC: 0.2411 ± 0.0052
Rata-rata best_iteration: 208.6


# train_xgboost 

In [14]:
result_xgb = train_xgboost(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_xgb["cv_summary"]["roc_auc_mean"], result_xgb["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_xgb["cv_summary"]["pr_auc_mean"], result_xgb["cv_summary"]["pr_auc_std"]
))
print("Rata-rata best_iteration:", np.mean([fm["best_iteration"] for fm in result_xgb["cv_fold_metrics"]]))

CV ROC-AUC: 0.7545 ± 0.0010
CV PR-AUC: 0.2388 ± 0.0029
Rata-rata best_iteration: 198.6


# train_catboost

In [15]:
result_cb = train_catboost(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_cb["cv_summary"]["roc_auc_mean"], result_cb["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_cb["cv_summary"]["pr_auc_mean"], result_cb["cv_summary"]["pr_auc_std"]
))
print("Rata-rata best_iteration:", np.mean([fm["best_iteration"] for fm in result_cb["cv_fold_metrics"]]))
print("n_features_after_encoding:", result_cb["pipeline"].named_steps["preprocessor"].transform(df[metadata["tree_features"]].head(5)).shape[1])

CV ROC-AUC: 0.7594 ± 0.0016
CV PR-AUC: 0.2436 ± 0.0059
Rata-rata best_iteration: 580.2
n_features_after_encoding: 66


# train_mlp

In [16]:
result_mlp = train_mlp(df, metadata, split, folds)

print("CV ROC-AUC: {:.4f} ± {:.4f}".format(
    result_mlp["cv_summary"]["roc_auc_mean"], result_mlp["cv_summary"]["roc_auc_std"]
))
print("CV PR-AUC: {:.4f} ± {:.4f}".format(
    result_mlp["cv_summary"]["pr_auc_mean"], result_mlp["cv_summary"]["pr_auc_std"]
))
print("Rata-rata n_iter (early stopping):", np.mean([fm["best_iteration"] for fm in result_mlp["cv_fold_metrics"]]))

CV ROC-AUC: 0.7469 ± 0.0006
CV PR-AUC: 0.2231 ± 0.0032
Rata-rata n_iter (early stopping): 14.2


# summary

In [17]:
display_model_comparison()

,Model,Jalur Fitur,N Fitur,ROC-AUC (mean),ROC-AUC (std),PR-AUC (mean),PR-AUC (std),Best Iter,Waktu Training (s)
rank,,,,,,,,,
1,catboost,tree,66,0.7594,0.0016,0.2436,0.0059,580.2,461.5
2,lightgbm,tree,185,0.7571,0.0010,0.2411,0.0052,208.6,20.2
3,xgboost,tree,185,0.7545,0.0010,0.2388,0.0029,198.6,39.8
4,logistic_regression,linear_mlp,185,0.7475,0.0020,0.2248,0.0060,-,28.2
5,mlp,linear_mlp,185,0.7469,0.0006,0.2231,0.0032,14.2,63.4
6,random_forest,tree,185,0.7441,0.0021,0.2187,0.0038,-,103.4
7,decision_tree,tree,185,0.5388,0.0052,0.0912,0.0020,-,45.5
